In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os, re, functools, json
import pandas as pd
import numpy as np

from src.scryfall_http_service import ScryfallHttpService
from src.scryfall_data_service import ScryfallDataService
from src.cache_service import CacheService, CacheType

pd.set_option('display.max_columns', None)

os.getcwd()

'c:\\Users\\tallo\\Documents\\programming_projects\\mtg-coding'

In [3]:
def get_commanders_from_txt_file(fpath: str) -> list[str]:
    with open(fpath, 'r') as f:
        return [x.strip() for x in f.read().split("\n")]


In [4]:
sfds = ScryfallDataService(
    base_url="https://api.scryfall.com/", 
    json_cache_service=CacheService("./cache/json/", cache_type=CacheType.JSON), 
    png_cache_service=CacheService("./cache/png/", cache_type=CacheType.PNG), 
    http_client=ScryfallHttpService(),
)

# sfds.clear_caches()

In [5]:
cards_df = sfds.get_cards_from_bulk_data('gameplay_columns')
cards_df.shape

(38233, 23)

In [ ]:
def get_type_to_subtype_dict(fpath: str) -> dict[str, list[str]]:
    with open("./data/types_to_subtypes.json", 'r') as f:
        return json.loads(f.read())

def get_types_from_type_line(type_line: str, valid_card_types: list[str]) -> list[str]:
    return sorted([x for x in valid_card_types if x.lower() in type_line.lower()])

def get_subtypes_from_type_line(type_line: str, type_to_subtype_dict: dict[str, list[str]]) -> list[str]:
    subtypes = []
    for k, v in type_to_subtype_dict.items():
        if k.lower() not in type_line.lower():
            continue
        for subtype in v:
            if subtype.lower() not in type_line.lower():
                continue
            subtypes.append(subtype.lower())
    return sorted(subtypes)

def get_supertypes_from_type_line(type_line: str) -> list[str]:
    supertypes = ["basic", "legendary", "ongoing", "snow", "world"]
    return sorted([x for x in supertypes if x in type_line.lower()])

def select_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    return df[[x for x in columns if x in df.columns]]

def transform_type_line(df: pd.DataFrame, type_to_subtype_dict: dict[str, list[str]]) -> pd.DataFrame:
    df['is_split_card'] = df['name'].apply(lambda x: "//" in x)
    df['supertypes'] = df['type_line'].apply(get_supertypes_from_type_line)
    df['types'] = df['type_line'].apply(functools.partial(get_types_from_type_line, valid_card_types=type_to_subtype_dict.keys()))
    df['subtypes'] = df['type_line'].apply(functools.partial(get_subtypes_from_type_line, type_to_subtype_dict=type_to_subtype_dict))
    return df

def drop_all_na_columns(df: pd.DataFrame) -> pd.DataFrame:
    cols_to_remove = []
    for c in df.columns:
        if df[~df[c].isna()].shape[0] > 0:
            continue 
        cols_to_remove.append(c)
    return df.drop(columns=cols_to_remove)

def rename_bool_cols(df: pd.DataFrame) -> pd.DataFrame:
    fix_col_name = lambda c: re.sub("^", "is_", c) if df[c].dtype == bool and len(re.findall("^is_", c)) == 0 else c
    df.columns = [fix_col_name(c) for c in df.columns]
    return df

def transform_oracle_text(df: pd.DataFrame) -> pd.DataFrame:
    df['oracle_lines'] = df['oracle_text'].apply(lambda x: x.split("\n") if isinstance(x, str) else x)
    return df

commander_df = cards_df[cards_df.is_commander_legal].copy()
commander_df = commander_df \
    .pipe(functools.partial(transform_type_line, type_to_subtype_dict=get_type_to_subtype_dict("./data/type_to_subtypes.json"))) \
    .pipe(drop_all_na_columns) \
    .pipe(rename_bool_cols) \
    .pipe(transform_oracle_text) \
    .pipe(functools.partial(select_columns, columns=['oracle_id', 'name', 'types', 'oracle_text', 'oracle_lines', 'keywords'])) \

subsample_df = commander_df[~commander_df.oracle_text.isna() & commander_df.keywords.str.len().ge(5)]
for idx, row in subsample_df.iterrows():
    print(row['name'])
    print(row['keywords'])
    print(row['oracle_text'])
    print(row['oracle_lines'])
    break

Cosmic Spider-Man
['Flying', 'Lifelink', 'First strike', 'Haste', 'Trample']
Flying, first strike, trample, lifelink, haste
At the beginning of combat on your turn, other Spiders you control gain flying, first strike, trample, lifelink, and haste until end of turn.
['Flying, first strike, trample, lifelink, haste', 'At the beginning of combat on your turn, other Spiders you control gain flying, first strike, trample, lifelink, and haste until end of turn.']
